In [3]:
#!/usr/bin/env python3
"""
Computes the "Cell Specific Painome" summary numbers directly from files
already written by step1/step2/step3/step2.5 — no need to rerun any of the
R scripts or MultiXrank.

    22 neuron cell types with one PPI backbone. X unique genes, X unique PPI
    edges, X unique between layer edges. Avg size of network X nodes, X edges
    408 genes converted to X on network, X most common gene on all layers

IMPORTANT: after step2.5_namespace_and_rebuild.py runs, gene_ppi_reference_
graph.pkl / gene_reference_graph.pkl and the multiplex/bipartite .tsv files
are all NAMESPACED ("celltype::GENE"). Counting PPI edges off the namespaced
files would count the same gene pair once per cell type it appears in,
instead of once overall. This script prefers the *.tsv.orig backups
(written by step2.5 before it namespaces) so "unique PPI edges" and "unique
genes" are true cross-layer-deduplicated counts. celltype_gene_sets.json and
cell_type_metadata.json are never touched by step2.5, so those are used
as-is.

Usage:
    python compute_painome_stats.py --wdir /path/to/multixrank_hetero_input \
        [--global-ppi-name global_ppi]
"""
import argparse
import json
from collections import Counter
from pathlib import Path

import pandas as pd


def read_edge_file(path: Path) -> set[frozenset]:
    df = pd.read_csv(path, sep="\t", header=None, names=["g1", "g2"], dtype=str)
    return {frozenset((a, b)) for a, b in zip(df["g1"], df["g2"]) if a != b}


def multiplex_edge_file(mul_root: Path, ct: str) -> Path:
    """Prefer the pre-namespacing backup (plain gene symbols) if it exists."""
    orig = mul_root / ct / "ppi_layer.tsv.orig"
    return orig if orig.exists() else mul_root / ct / "ppi_layer.tsv"


def bipartite_edge_file(wdir: Path, rel_file: str) -> Path:
    p = wdir / rel_file
    orig = p.with_suffix(".tsv.orig")
    return orig if orig.exists() else p


wdir="04_multixrank_prep/multixrank_hetero_input"
global_ppi_name="global_ppi"
wdir = Path(wdir)
mul_root = wdir / "multiplex"

# --- gene universe + most common gene, from celltype_gene_sets.json ---
with open(wdir / "celltype_gene_sets.json") as f:
    gene_sets = json.load(f)  # cell_type -> list of genes (untouched by step2.5)

neuron_cts = [ct for ct in gene_sets if ct != global_ppi_name]
all_genes = set()
gene_layer_count = Counter()
for ct, genes in gene_sets.items():
    all_genes.update(genes)
    for g in genes:
        gene_layer_count[g] += 1

n_unique_genes = len(all_genes)
most_common_gene, most_common_count = gene_layer_count.most_common(1)[0]

# --- unique PPI edges (global dedup across layers), from .orig backups ---
all_ppi_edges = set()
for ct in gene_sets:
    f = multiplex_edge_file(mul_root, ct)
    if f.exists():
        all_ppi_edges |= read_edge_file(f)
n_unique_ppi_edges = len(all_ppi_edges)

# --- avg network size, from cell_type_metadata.json (neuron layers only) ---
with open(wdir / "cell_type_metadata.json") as f:
    metadata = json.load(f)
neuron_meta = {ct: m for ct, m in metadata.items() if ct != global_ppi_name}
if neuron_meta:
    avg_nodes = sum(m["n_nodes"] for m in neuron_meta.values()) / len(neuron_meta)
    avg_edges = sum(m["n_ppi_edges"] for m in neuron_meta.values()) / len(neuron_meta)
else:
    avg_nodes = avg_edges = float("nan")

# --- unique between-layer edges, from bipartite_manifest.json ---
manifest_path = wdir / "bipartite_manifest.json"
n_between_layer_edges = 0
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    n_between_layer_edges = sum(m["n_edges"] for m in manifest)

# --- seeds: 408 raw -> X present on network ---
seeds_dir = wdir / "seeds"
n_raw_seeds = None
n_seeds_on_network = None
raw_path = seeds_dir / "seeds_original.txt"
if raw_path.exists():
    n_raw_seeds = len([l for l in raw_path.read_text().splitlines() if l.strip()])

plain_backup = seeds_dir / "seeds_plain_from_step3.txt"
if plain_backup.exists():
    # step2.5 has run: this backup is the deduped, network-filtered,
    # PRE-expansion seed list -- exactly "408 genes converted to X".
    n_seeds_on_network = len(
        {l.strip().upper() for l in plain_backup.read_text().splitlines() if l.strip()}
    )
else:
    # step2.5 hasn't run yet -- current seeds.txt (written by step3) IS
    # already the plain, network-filtered list.
    cur = seeds_dir / "seeds.txt"
    if cur.exists():
        lines = [l.strip() for l in cur.read_text().splitlines() if l.strip()]
        if not any("::" in l for l in lines):
            n_seeds_on_network = len(set(lines))

# --- is the most common gene a seed? ---
seed_gene_set = set()
plain_backup = seeds_dir / "seeds_plain_from_step3.txt"
raw_path = seeds_dir / "seeds_original.txt"
if plain_backup.exists():
    seed_gene_set = {l.strip().upper() for l in plain_backup.read_text().splitlines() if l.strip()}
elif raw_path.exists():
    seed_gene_set = {l.strip().upper() for l in raw_path.read_text().splitlines() if l.strip()}
most_common_is_seed = most_common_gene.upper() in seed_gene_set if seed_gene_set else None

# --- PPI validation check on between-layer edges ---
ppi_check_path = wdir / "ppi_check_stats.csv"
ppi_check_df = pd.read_csv(ppi_check_path) if ppi_check_path.exists() else None

# --- sanity check: union must be >= global_ppi alone ---
gp_name = global_ppi_name
gp_nodes = metadata.get(gp_name, {}).get("n_nodes")
gp_edges = metadata.get(gp_name, {}).get("n_ppi_edges")
union_smaller_than_backbone = (
    gp_name in gene_sets and gp_nodes is not None and
    (n_unique_genes < gp_nodes or n_unique_ppi_edges < (gp_edges or 0))
)

# ------------------------------------------------------------------
print("=" * 70)
print("CELL SPECIFIC PAINOME — SUMMARY")
print("=" * 70)
print(f"Neuron cell types found       : {len(neuron_cts)}")
print(f"Backbone multiplex present     : {global_ppi_name in gene_sets} "
      f"({global_ppi_name})")
print()
print(f"Unique genes (all layers)      : {n_unique_genes:,}")
print(f"Unique PPI edges (all layers)  : {n_unique_ppi_edges:,}")
print(f"Unique between-layer edges     : {n_between_layer_edges:,}  "
      f"(includes ligand-receptor/pathway edges + backbone identity edges)")
print()
print(f"Avg network size (neuron layers only, n={len(neuron_meta)}):")
print(f"  nodes : {avg_nodes:,.1f}")
print(f"  edges : {avg_edges:,.1f}")
print()
if n_raw_seeds is not None:
    print(f"Seeds provided                 : {n_raw_seeds}")
if n_seeds_on_network is not None:
    print(f"Seeds converted to on-network   : {n_seeds_on_network}")
print()
print(f"Most common gene across layers : {most_common_gene} "
      f"(present in {most_common_count}/{len(gene_sets)} layers)")
if most_common_is_seed is None:
    print(f"  Is it a seed?                 : couldn't tell (no seeds file found)")
else:
    print(f"  Is it a seed?                 : {most_common_is_seed}")
print()

if gp_name in gene_sets and gp_nodes is not None:
    print(f"Backbone ('{gp_name}') alone     : {gp_nodes:,} nodes, {gp_edges:,} edges")
    if union_smaller_than_backbone:
        print(f"  ⚠ Your all-layer union is SMALLER than the backbone alone. If your "
              f"'unique genes/edges' totals above are meant to include {gp_name}, this "
              f"shouldn't be possible mathematically -- double check you're reading the "
              f"union from the same files as this script (celltype_gene_sets.json / "
              f".tsv.orig) rather than a stale or filtered snapshot. If instead your totals "
              f"deliberately excluded {gp_name} (e.g. neuron layers only), a smaller number "
              f"here is expected -- the backbone is a comprehensive, expression-agnostic "
              f"PPI reference and isn't restricted to genes detected in your 22 cell types.")
    else:
        print(f"  Union totals are >= backbone alone, as expected.")
print()

if ppi_check_df is not None:
    print("PPI validation check (from ppi_check_stats.csv):")
    print(ppi_check_df.to_string(index=False))
    print("  NOTE: this diagnostic covers ligand_receptor/reactome/liana/neuronchat "
          f"candidate pairs only. '{gp_name}' identity edges (backbone <-> cell type) are "
          "never checked against a PPI reference -- they're gene-presence links, not "
          "asserted PPI/LR relationships, by design.")
else:
    print(f"PPI validation check: {ppi_check_path.name} not found in {wdir} -- "
          f"either step2 was run without --ppi-pkl, or you're pointed at the wrong wdir.")
print("=" * 70)

print("\n--- Fill-in-the-blank version ---\n")
print(f"{len(neuron_cts)} neuron cell types with one PPI backbone. "
      f"{n_unique_genes:,} unique genes, {n_unique_ppi_edges:,} unique PPI edges, "
      f"{n_between_layer_edges:,} unique between layer edges. "
      f"Avg size of network {avg_nodes:,.0f} nodes, {avg_edges:,.0f} edges")
seeds_str = f"{n_raw_seeds}" if n_raw_seeds is not None else "408"
onnet_str = f"{n_seeds_on_network}" if n_seeds_on_network is not None else "?"
print(f"{seeds_str} genes converted to {onnet_str} on network, "
      f"{most_common_gene} most common gene on all layers")

CELL SPECIFIC PAINOME — SUMMARY
Neuron cell types found       : 22
Backbone multiplex present     : True (global_ppi)

Unique genes (all layers)      : 17,051
Unique PPI edges (all layers)  : 302,553
Unique between-layer edges     : 137,380  (includes ligand-receptor/pathway edges + backbone identity edges)

Avg network size (neuron layers only, n=22):
  nodes : 2,738.7
  edges : 11,603.5

Seeds provided                 : 408
Seeds converted to on-network   : 372

Most common gene across layers : ADGRG2 (present in 23/23 layers)
  Is it a seed?                 : False

Backbone ('global_ppi') alone     : 17,051 nodes, 302,553 edges
  Union totals are >= backbone alone, as expected.

PPI validation check (from ppi_check_stats.csv):
         source  before_ppi_check  after_ppi_check  pct_ppi_corroborated
          liana              1856              249                  13.4
ligand_receptor              2491              423                  17.0
     neuronchat                39       

In [5]:
#!/usr/bin/env python3
"""
Diagnoses exactly why global_ppi's node/edge counts in cell_type_metadata.json
(written by step1_build_celltype_ppi.py) differ from the raw --base-ppi pickle
you originally passed in.

step1 does two things to the raw pickle before counting anything:
  1. load_base_ppi() uppercases every node label via
         nx.relabel_nodes(G, {n: str(n).strip().upper() for n in G.nodes()})
     If two node labels differ only in case (e.g. "Cxcl12" / "CXCL12"), this
     merges them into one node and unions their edges -- nx.Graph has no
     parallel edges, so any edge that existed from BOTH case-variants to the
     same neighbor collapses into a single edge, silently shrinking both
     node and edge counts.
  2. write_ppi_layer() drops self-loops via `if a != b` before writing/
     counting -- any gene-A<->gene-A self-interaction in the raw pickle
     disappears from n_ppi_edges without a trace.

This script loads your RAW pickle directly (no relabeling) and reports:
  - raw node/edge counts
  - which node labels are case-duplicates of each other, and how many
    distinct nodes/edges that collapses away
  - how many self-loop edges exist
  - a simulated post-step1 count (uppercase + drop self-loops), which
    should exactly match cell_type_metadata.json's global_ppi entry

Usage:
    python diagnose_base_ppi_gap.py --ppi-pkl /path/to/base_ppi.pkl \
        [--metadata /path/to/cell_type_metadata.json] [--global-ppi-name global_ppi]
"""
import argparse
import json
import pickle
from collections import defaultdict
from pathlib import Path

import networkx as nx


ppi_pkl="00_inputs/ppi_network.pkl"
metadata=" 04_multixrank_prep/multixrank_hetero_input/cell_type_metadata.json"
global_ppi_name="global_ppi"

print(f"Loading raw PPI pickle: {ppi_pkl} ...")
with open(ppi_pkl, "rb") as f:
    G_raw = pickle.load(f)

raw_nodes = G_raw.number_of_nodes()
raw_edges = G_raw.number_of_edges()
print(f"\nRAW file (exactly as loaded, no processing):")
print(f"  nodes : {raw_nodes:,}")
print(f"  edges : {raw_edges:,}")

# --- self-loops in the raw file ---
self_loops = [(u, v) for u, v in G_raw.edges() if u == v]
print(f"\nSelf-loop edges (gene <-> itself) in raw file: {len(self_loops):,}")
if self_loops:
    print(f"  Examples: {self_loops[:10]}")

# --- case-duplicate node labels ---
upper_groups = defaultdict(list)
for n in G_raw.nodes():
    upper_groups[str(n).strip().upper()].append(n)
dup_groups = {u: labels for u, labels in upper_groups.items() if len(labels) > 1}
n_nodes_collapsed = sum(len(labels) - 1 for labels in dup_groups.values())

print(f"\nCase-duplicate node labels (distinct raw labels that uppercase to the same gene):")
print(f"  {len(dup_groups)} gene(s) affected, {n_nodes_collapsed} extra node(s) that will "
      f"collapse away when uppercased")
if dup_groups:
    shown = list(dup_groups.items())[:10]
    for u, labels in shown:
        print(f"    {u} <- {labels}")
    if len(dup_groups) > 10:
        print(f"    ... and {len(dup_groups) - 10} more")

# --- simulate exactly what load_base_ppi() + write_ppi_layer() do ---
print("\nSimulating step1's processing (uppercase relabel, then drop self-loops) ...")
mapping = {n: str(n).strip().upper() for n in G_raw.nodes()}
G_processed = nx.relabel_nodes(G_raw, mapping, copy=True)
processed_nodes_with_loops = G_processed.number_of_nodes()
processed_edges_with_loops = G_processed.number_of_edges()

final_edges = [(a, b) for a, b in G_processed.edges() if a != b]
final_nodes = processed_nodes_with_loops
final_edge_count = len(final_edges)

print(f"\nAfter uppercase relabel (before dropping self-loops):")
print(f"  nodes : {processed_nodes_with_loops:,}  (raw {raw_nodes:,} -> "
      f"{raw_nodes - processed_nodes_with_loops:,} collapsed)")
print(f"  edges : {processed_edges_with_loops:,}  (raw {raw_edges:,} -> "
      f"{raw_edges - processed_edges_with_loops:,} merged away)")

print(f"\nAfter also dropping self-loops (this is what n_ppi_edges in "
      f"cell_type_metadata.json actually reflects):")
print(f"  nodes : {final_nodes:,}")
print(f"  edges : {final_edge_count:,}")

print("\n" + "=" * 70)
print("SUMMARY: raw -> what step1 actually wrote")
print("=" * 70)
print(f"  Nodes: {raw_nodes:,} raw -> {final_nodes:,} written "
      f"({raw_nodes - final_nodes:,} lost to case-duplicate merging)")
print(f"  Edges: {raw_edges:,} raw -> {final_edge_count:,} written "
      f"({raw_edges - final_edge_count:,} lost: "
      f"{len(self_loops)} self-loops dropped, "
      f"{raw_edges - final_edge_count - len(self_loops)} merged away by case-collapsing)")

if metadata:
    meta_path = Path(metadata)
    if meta_path.exists():
        with open(meta_path) as f:
            metadata = json.load(f)
        gp = metadata.get(global_ppi_name)
        if gp:
            print(f"\nCompare against {meta_path} ['{global_ppi_name}']:")
            print(f"  n_nodes     : {gp['n_nodes']:,}  "
                  f"(simulated: {final_nodes:,}, "
                  f"{'MATCH' if gp['n_nodes'] == final_nodes else 'MISMATCH -- something else is going on'})")
            print(f"  n_ppi_edges : {gp['n_ppi_edges']:,}  "
                  f"(simulated: {final_edge_count:,}, "
                  f"{'MATCH' if gp['n_ppi_edges'] == final_edge_count else 'MISMATCH -- something else is going on'})")
        else:
            print(f"\n'{global_ppi_name}' not found in {meta_path}")
    else:
        print(f"\nMetadata file not found: {meta_path}")
print("=" * 70)


Loading raw PPI pickle: 00_inputs/ppi_network.pkl ...

RAW file (exactly as loaded, no processing):
  nodes : 17,053
  edges : 302,564

Self-loop edges (gene <-> itself) in raw file: 0

Case-duplicate node labels (distinct raw labels that uppercase to the same gene):
  2 gene(s) affected, 2 extra node(s) that will collapse away when uppercased
    RNPS1 <- ['RNPS1', 'rnps1']
    APOL6 <- ['ApoL6', 'APOL6']

Simulating step1's processing (uppercase relabel, then drop self-loops) ...

After uppercase relabel (before dropping self-loops):
  nodes : 17,051  (raw 17,053 -> 2 collapsed)
  edges : 302,553  (raw 302,564 -> 11 merged away)

After also dropping self-loops (this is what n_ppi_edges in cell_type_metadata.json actually reflects):
  nodes : 17,051
  edges : 302,553

SUMMARY: raw -> what step1 actually wrote
  Nodes: 17,053 raw -> 17,051 written (2 lost to case-duplicate merging)
  Edges: 302,564 raw -> 302,553 written (11 lost: 0 self-loops dropped, 11 merged away by case-collapsing